# 🐍 Day 36 · Python Basics — Modules, Packages & Agent Configs (Google ADK)

> **Goal of this notebook:** Each ADK agent is a **package**: a folder with `__init__.py`, `agent.py`, `prompt.py` and `tools/`. Learn how Python packages and imports work by building one.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. What a module and a package are
2. Creating a small package on disk
3. How Python finds imports (`sys.path`)
4. `from package.module import name`
5. `if __name__ == "__main__":`
6. 🔗 An agent described as data (name, model, instruction, tools)

## 1. Module vs package
- A **module** = one `.py` file (`tools.py`).
- A **package** = a folder of modules with an `__init__.py` file.

Look at `adk_simple_bot/` in this folder:
```
adk_simple_bot/
├── __init__.py        <- makes it a package
├── agent.py           <- defines root_agent
├── prompt.py          <- the instructions
└── tools/
    ├── __init__.py
    └── wether_tool.py
```

## 2. Build a tiny package on disk

In [ ]:
from pathlib import Path

pkg = Path("demo_bot")
(pkg / "tools").mkdir(parents=True, exist_ok=True)

(pkg / "__init__.py").write_text("from . import agent\n", encoding="utf-8")
(pkg / "tools" / "__init__.py").write_text("from .weather import get_weather\n", encoding="utf-8")
(pkg / "tools" / "weather.py").write_text(
    'def get_weather(city: str) -> dict:\n'
    '    """Return a pretend weather report for a city."""\n'
    '    return {"city": city, "report": "sunny, 28°C"}\n'
, encoding="utf-8")
(pkg / "prompt.py").write_text('INSTRUCTION = "You are a friendly weather assistant."\n', encoding="utf-8")
(pkg / "agent.py").write_text(
    "from demo_bot.prompt import INSTRUCTION\n"
    "from demo_bot.tools import get_weather\n\n"
    "root_agent = {'name': 'demo_bot', 'model': 'gemini-2.5-flash',\n"
    "              'instruction': INSTRUCTION, 'tools': [get_weather]}\n"
, encoding="utf-8")
for p in sorted(pkg.rglob("*.py")):
    print(p)

## 3. How Python finds imports
Python looks through the folders in `sys.path`. The folder you run from is (usually) the first one — that's why we always `cd` into the Day folder before running.

In [ ]:
import sys, os
print(sys.path[:3])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())      # make sure this folder is searched

## 4. Import from the package

In [ ]:
from demo_bot.agent import root_agent
from demo_bot.tools import get_weather

print(root_agent["name"], "uses", root_agent["model"])
print(get_weather("Bengaluru"))
print("Tool description:", get_weather.__doc__)

## 5. `if __name__ == "__main__":`
Code inside this block runs only when the file is **run directly** (`python file.py`), not when it is **imported**. You see it at the bottom of many scripts in this course.

In [ ]:
Path("demo_script.py").write_text(
    'def hello():\n    return "hi from a function"\n\n'
    'if __name__ == "__main__":\n    print("running directly!", hello())\n'
, encoding="utf-8")
import subprocess
print(subprocess.run([sys.executable, "demo_script.py"], capture_output=True, text=True).stdout)   # run it directly

import demo_script                       # import it: nothing is printed
print("imported, __name__ is:", demo_script.__name__)

## 6. 🔗 An agent described as data
In ADK you write `Agent(name=..., model=..., instruction=..., tools=[...])`. A dataclass shows the same idea.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AgentConfig:
    name: str
    model: str
    instruction: str
    tools: list = field(default_factory=list)
    sub_agents: list = field(default_factory=list)

researcher = AgentConfig("researcher", "gemini-2.5-flash", "Find facts about the topic.")
writer = AgentConfig("scriptwriter", "gemini-2.5-flash", "Write a podcast script.")
podcast = AgentConfig("podcast_agent", "gemini-2.5-pro", "Coordinate the team.", sub_agents=[researcher, writer])

print(podcast.name, "->", [a.name for a in podcast.sub_agents])

In [ ]:
# Clean up the demo files and forget the imported modules
import shutil
shutil.rmtree(pkg); os.remove("demo_script.py")
for f in Path("__pycache__").glob("demo_script*.pyc"):   # Python saved a compiled copy when we imported it
    f.unlink()
for name in [m for m in sys.modules if m.startswith(("demo_bot", "demo_script"))]:
    del sys.modules[name]
print("demo files deleted ✅")

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Which file makes a folder a Python package?
2. Add a second tool `get_time(city)` to the demo package (re-run section 2 first).
3. Create an `AgentConfig` for an `"audio_producer"` and add it to `podcast.sub_agents`.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
# 1. __init__.py

# 3.
podcast.sub_agents.append(AgentConfig("audio_producer", "gemini-2.5-flash", "Turn the script into audio."))
print([a.name for a in podcast.sub_agents])
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Package | `folder + __init__.py` |
| Import from package | `from demo_bot.tools import get_weather` |
| Search path | `sys.path` |
| Run only directly | `if __name__ == '__main__':` |
| Agent as data | `AgentConfig(name, model, instruction, tools)` |

## ➡️ What's next?
You now know the Python used in **Day_36_Google_ADK_Agents**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀